# Бустинги и ансамбли для анализа конверсии

Предсказываем `Revenue` по завершённой сессии Online Shoppers.
Сравниваем библиотеки, подбираем параметры и оцениваем влияние дисбаланса.

## Содержание

1. [Данные](#data)
2. [Бустинги](#boosting)
3. [Optuna](#optuna)
4. [Дисбаланс](#imbalance)
5. [Стекинг](#stacking)
6. [Итоговая оценка](#evaluation)
7. [SHAP](#shap)
8. [Сохранение](#saving)

In [ ]:
import pickle
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import shap
from IPython.display import display, Markdown
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

SEED = 42
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
MODEL_PATH = ROOT / "models/session_advanced_model.pkl"
CAT = ["Month", "OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType", "Weekend"]
optuna.logging.set_verbosity(optuna.logging.WARNING)

<a id="data"></a>
## Данные

Берём исходный CSV без искусственных пропусков. Полные совпадения признаков
остаются в одной группе при разделении и кросс-валидации. Тест – около 20% групп.
Остальные данные используем для выбора моделей на пяти одинаковых фолдах.

In [ ]:
data = pd.read_csv(ROOT / "data/raw/online_shoppers_intention.csv")
X = data.drop(columns="Revenue")
y = data.Revenue.astype(int)
NUM = [c for c in X if c not in CAT]

def folds_for(frame, target):
    groups = pd.factorize(pd.MultiIndex.from_frame(frame))[0]
    return list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(frame, target, groups))

groups = pd.factorize(pd.MultiIndex.from_frame(X))[0]
unique = pd.DataFrame({"group": groups, "target": y})
if unique.groupby("group").target.nunique().max() > 1:
    raise ValueError("Совпадающие признаки имеют разные цели.")
unique = unique.drop_duplicates("group")
train_groups, test_groups = train_test_split(unique.group, stratify=unique.target, test_size=0.2, random_state=SEED)
mask = np.isin(groups, train_groups)
X_train, X_test = X.loc[mask], X.loc[~mask]
y_train, y_test = y.loc[mask], y.loc[~mask]
folds = folds_for(X_train, y_train)
display(pd.DataFrame({"строк": [len(X_train), len(X_test)], "доля покупок": [y_train.mean(), y_test.mean()]}, index=["train", "test"]))
print("Пропусков:", int(X.isna().sum().sum()), "повторов:", int(X.duplicated().sum()))

In [ ]:
def prepare(scale=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale: numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("num", SkPipeline(numeric_steps), NUM),
        ("cat", SkPipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CAT),
    ])

def boosting(name, params=None, balance="none", target=None):
    settings = dict(params or {})
    ratio = float((target == 0).sum() / (target == 1).sum()) if balance == "weights" else 1.0
    if name == "XGBoost":
        model = XGBClassifier(random_state=SEED, n_jobs=2, **settings, **({"scale_pos_weight": ratio} if balance == "weights" else {}))
    elif name == "LightGBM":
        model = LGBMClassifier(random_state=SEED, n_jobs=2, verbosity=-1, **settings, **({"scale_pos_weight": ratio} if balance == "weights" else {}))
    else:
        model = CatBoostClassifier(random_seed=SEED, thread_count=2, verbose=False, allow_writing_files=False, **settings, **({"class_weights": [1.0, ratio]} if balance == "weights" else {}))
    return Pipeline([
        ("preprocessor", prepare()),
        ("sampler", RandomUnderSampler(random_state=SEED) if balance == "under" else "passthrough"),
        ("model", model),
    ])

def metrics(target, probability):
    prediction = probability >= 0.5
    return {"ROC-AUC": roc_auc_score(target, probability), "AP": average_precision_score(target, probability),
            "F1": f1_score(target, prediction, zero_division=0), "Precision": precision_score(target, prediction, zero_division=0),
            "Recall": recall_score(target, prediction, zero_division=0)}

all_rows = []
def evaluate_cv(label, factory):
    records = []
    for fit, valid in folds:
        model = factory(X_train.iloc[fit], y_train.iloc[fit])
        started = time.perf_counter()
        model.fit(X_train.iloc[fit], y_train.iloc[fit])
        elapsed = time.perf_counter() - started
        row = metrics(y_train.iloc[valid], model.predict_proba(X_train.iloc[valid])[:, 1])
        row["Fit seconds"] = elapsed
        records.append(row)
    scores = pd.DataFrame(records)
    result = {"Модель": label, **scores.mean().to_dict(), "ROC-AUC std": scores["ROC-AUC"].std(ddof=1)}
    all_rows.append(result)
    print(label, round(result["ROC-AUC"], 5), flush=True)
    return result

<a id="boosting"></a>
## Бустинги

Используем параметры обучения по умолчанию. Фиксируем seed, два потока CPU и отключаем
служебный вывод. Предобработка общая: медиана и one-hot. Деревьям масштабирование не нужно.
Это сравнение библиотек с одинаковой предобработкой, без нативных категорий CatBoost.
Стандартное число деревьев и бюджеты обучения у библиотек различаются.

In [ ]:
baseline_rows = []
for name in ["XGBoost", "LightGBM", "CatBoost"]:
    baseline_rows.append(evaluate_cv(name, lambda frame, target, name=name: boosting(name)))
baselines = pd.DataFrame(baseline_rows).set_index("Модель")
display(baselines.round(4))
best_library = baselines["ROC-AUC"].idxmax()
print("Выбрана библиотека:", best_library)

<a id="optuna"></a>
## Optuna

Проводим 50 полных trials с пятью фолдами внутри objective. Выбираем по среднему ROC-AUC.
Тест не участвует. Диапазоны ограничены, чтобы сравнение можно было повторить на CPU.

In [ ]:
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
    }
    count = trial.suggest_int("trees", 100, 400, step=50)
    depth = trial.suggest_int("depth", 3, 7)
    if best_library == "CatBoost":
        params.update(iterations=count, depth=depth, l2_leaf_reg=trial.suggest_float("regularization", 1, 20, log=True))
    elif best_library == "LightGBM":
        params.update(n_estimators=count, max_depth=depth, num_leaves=trial.suggest_int("num_leaves", 7, min(63, 2**depth)), reg_lambda=trial.suggest_float("regularization", 0.01, 10, log=True))
    else:
        params.update(n_estimators=count, max_depth=depth, reg_lambda=trial.suggest_float("regularization", 0.01, 10, log=True))
    trial.set_user_attr("model_params", params)
    values = []
    for fit, valid in folds:
        model = boosting(best_library, params)
        model.fit(X_train.iloc[fit], y_train.iloc[fit])
        values.append(roc_auc_score(y_train.iloc[valid], model.predict_proba(X_train.iloc[valid])[:, 1]))
    return float(np.mean(values))

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
def progress(study, trial):
    if (trial.number + 1) % 5 == 0: print("Trials:", trial.number + 1, "лучший ROC-AUC:", round(study.best_value, 5), flush=True)
study.optimize(objective, n_trials=50, n_jobs=1, callbacks=[progress])
best_params = study.best_trial.user_attrs["model_params"]
print("Лучшие параметры:", best_params)
print("ROC-AUC:", study.best_value)
plot_optimization_history(study)
plt.tight_layout(); plt.show()
plot_param_importances(study)
plt.tight_layout(); plt.show()
evaluate_cv("Optimized", lambda frame, target: boosting(best_library, best_params))

<a id="imbalance"></a>
## Дисбаланс

Сравниваем оптимизированный бустинг без коррекции, с весами и RandomUnderSampler.
Веса рассчитываются на train каждого фолда. Ресемплинг выполняется внутри imblearn Pipeline,
валидация сохраняет исходные доли классов. AP – Average Precision по PR-кривой.
F1 считается при одном пороге 0,5, без подбора на тесте.

In [ ]:
for balance in ["weights", "under"]:
    evaluate_cv(balance, lambda frame, target, balance=balance: boosting(best_library, best_params, balance, target))

<a id="stacking"></a>
## Стекинг

LightGBM, RandomForest и SVC дают вероятности для LogisticRegression.
Каждая базовая модель имеет свой препроцессор, обучаемый внутри внутренних фолдов.
Масштабирование нужно только SVC. Внешние пять фолдов оценивают весь ансамбль;
внутренние пять создают прогнозы для метамодели. Одинаковые сессии не пересекают фолды.

In [ ]:
def stacking(frame, target):
    light_params = best_params if best_library == "LightGBM" else {}
    estimators = [
        ("lightgbm", boosting("LightGBM", light_params)),
        ("forest", SkPipeline([("preprocessor", prepare()), ("model", RandomForestClassifier(n_estimators=150, min_samples_leaf=2, random_state=SEED, n_jobs=2))])),
        ("svc", SkPipeline([("preprocessor", prepare(scale=True)), ("model", SVC(probability=True, random_state=SEED))])),
    ]
    return StackingClassifier(estimators, final_estimator=LogisticRegression(max_iter=2000, random_state=SEED),
                              cv=folds_for(frame, target), stack_method="predict_proba", n_jobs=1)

evaluate_cv("Stacking", stacking)
comparison = pd.DataFrame(all_rows).set_index("Модель")
display(comparison.sort_values("ROC-AUC", ascending=False).round(4))

<a id="evaluation"></a>
## Итоговая оценка

Выбираем среди оптимизированного бустинга, его двух вариантов коррекции и стекинга
по ROC-AUC кросс-валидации. Тест используем после выбора, сохраняя все его строки.

In [ ]:
candidates = comparison.loc[["Optimized", "weights", "under", "Stacking"]]
final_name = candidates["ROC-AUC"].idxmax()
if final_name == "Stacking":
    final_model = stacking(X_train, y_train)
else:
    balance = "none" if final_name == "Optimized" else final_name
    final_model = boosting(best_library, best_params, balance, y_train)
final_model.fit(X_train, y_train)
test_probability = final_model.predict_proba(X_test)[:, 1]
print("Итоговая модель:", final_name)
display(pd.Series(metrics(y_test, test_probability), name="Тест").round(4))
display(pd.DataFrame(confusion_matrix(y_test, test_probability >= 0.5), index=["Нет покупки", "Покупка"], columns=["Прогноз: нет", "Прогноз: да"]))

<a id="shap"></a>
## SHAP

Объясняем оптимизированный бустинг без коррекции, чтобы сохранить отдельный анализ
его признаков. Если победил стекинг, эти объяснения не относятся ко всему ансамблю.
Используем до 400 случайных строк train. SHAP описывает прогноз, а не причинное влияние.

In [ ]:
explained_model = boosting(best_library, best_params).fit(X_train, y_train)
preprocessor = explained_model.named_steps["preprocessor"]
sample = X_train.sample(n=min(400, len(X_train)), random_state=SEED)
encoded = preprocessor.transform(sample)
names = preprocessor.get_feature_names_out()
explainer = shap.TreeExplainer(explained_model.named_steps["model"], feature_names=names)
explanation = explainer(encoded)
if explanation.values.ndim == 3:
    explanation = explanation[:, :, 1]
shap.plots.beeswarm(explanation, max_display=15, show=False)
plt.tight_layout(); plt.show()
shap.plots.bar(explanation, max_display=15, show=False)
plt.tight_layout(); plt.show()
importance = np.abs(explanation.values).mean(axis=0)
top_two = np.argsort(-importance)[:2]
print("Два ведущих признака:", names[top_two].tolist())
for index in top_two:
    shap.plots.scatter(explanation[:, index], show=False)
    plt.tight_layout(); plt.show()
local_probability = explained_model.predict_proba(sample)[:, 1]
errors = np.flatnonzero((local_probability >= 0.5) != y_train.loc[sample.index].to_numpy())
objects = list(dict.fromkeys([int(np.argmax(local_probability)), int(np.argmin(local_probability)), int(errors[0]) if len(errors) else 0]))
for index in objects:
    print("Строка:", sample.index[index], "Revenue:", int(y_train.loc[sample.index[index]]), "P:", round(local_probability[index], 4))
    shap.plots.waterfall(explanation[index], max_display=10, show=False)
    plt.tight_layout(); plt.show()

### Что видно по SHAP

Главный признак – `PageValues`: большие значения обычно сдвигают прогноз в сторону покупки.
Нулевые значения чаще дают отрицательный вклад. Признак отражает ценность посещённых страниц;
его доступность до покупки требует отдельной проверки.

Второй признак – one-hot `Month_May`: май в этой выборке обычно снижает оценку покупки.
Ноябрь чаще повышает её. Это наблюдение о модели и составе данных, а не причинный эффект месяца.
Важность отдельного one-hot столбца не равна суммарной важности исходного `Month`.

На waterfall положительные вклады повышают raw score CatBoost, отрицательные снижают.
Шкала – логарифм шансов; вероятность получается через sigmoid.
В ошибочном прогнозе строки 7793 высокий `PageValues` сдвинул прогноз до P=0,5066,
хотя фактической покупки не было. Другие признаки частично компенсировали этот вклад.
Объекты выбраны среди train для объяснения, эти иллюстрации не оценивают обобщение.

## Выводы

In [ ]:
display(Markdown(
    f"Базовый лидер – **{best_library}**. Optuna: **{len(study.trials)} trials**, "
    f"лучший средний ROC-AUC **{study.best_value:.4f}**. "
    f"После сравнения коррекции дисбаланса и стекинга выбран **{final_name}**. "
    f"На тесте ROC-AUC **{metrics(y_test, test_probability)['ROC-AUC']:.4f}**, "
    f"AP **{metrics(y_test, test_probability)['AP']:.4f}**."
))

Сравнение последовательное, без внешней вложенной оценки всего подбора Optuna.
Средние по фолдам использованы для выбора, поэтому итоговая оценка берётся с теста.
Небольшие различия нужно сопоставлять с разбросом, а не считать гарантированным улучшением.

Стекинг дороже одиночной модели из-за внутренних фолдов и калибровки SVC.
Внутренняя калибровка вероятностей SVC управляется libsvm; она не использует наши группы.
При RandomUnderSampler теряется часть объектов большинства, а веса меняют цену ошибок.
Вероятности после коррекции дисбаланса не проверялись на калибровку.

`PageValues` и итоговые показатели сессии могут быть недоступны до покупки.
Результат относится к завершённым сессиям; случайное разделение не измеряет перенос на будущие месяцы.

### Сравнение результатов

CatBoost дал лучший базовый ROC-AUC 0,9320, LightGBM – 0,9298, XGBoost – 0,9208.
LightGBM обучался быстрее: около 0,36 секунды на фолд против 2,41 у CatBoost.
Для дальнейшей работы выбран CatBoost по качеству. Его преимущество невелико
относительно разброса по фолдам, поэтому это не доказательство превосходства на любых данных.

Optuna подняла средний ROC-AUC до 0,9340 и AP до 0,7549; выбранная модель с 350 деревьями
обучалась около 0,78 секунды на фолд. Лучшие параметры найдены в заданном диапазоне,
а не среди всех возможных конфигураций.

При весах recall вырос с 0,5895 до 0,8181, precision снизился с 0,7336 до 0,5543.
F1 увеличился с 0,6533 до 0,6607, но ROC-AUC и AP немного снизились.
RandomUnderSampler дал recall 0,8550 при precision 0,5280 и AP 0,7355.
Это обмен точности на полноту, а не однозначное улучшение.

Стекинг получил ROC-AUC 0,9284 и занимал около 31,87 секунды на внешний фолд.
По выбранному критерию усложнение не оправдалось. Сохранён CatBoost без коррекции дисбаланса.
На 2 469 тестовых сессиях ROC-AUC 0,9373, AP 0,7500, F1 0,6610.
Порог 0,5 пропустил 149 покупок и дал 90 ложных срабатываний.

<a id="saving"></a>
## Сохранение

Сохраняем выбранный полный конвейер через стандартный pickle.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
with MODEL_PATH.open("wb") as stream:
    pickle.dump(final_model, stream, protocol=pickle.HIGHEST_PROTOCOL)
print(MODEL_PATH.relative_to(ROOT))